# 05 - The Integer Hull

**Goal:** Understand how the integer hull turns a discrete feasible set into a convex polyhedron.

Integer optimization starts with a polyhedron but only allows selected variables to take integer values. The feasible integer points are discrete and therefore usually non-convex. In this notebook we will:

1. Define integer and mixed-integer hulls
2. Construct the integer hull of a small 2D polytope
3. Compare the LP relaxation, integer feasible points, and integer hull
4. See how a new valid inequality removes a fractional LP optimum
5. Explain why algorithms usually approximate the integer hull instead of constructing it explicitly

The computational examples use enumeration and convex hulls. This is ideal for learning in two dimensions, but not a method for large integer programs.

## From a polyhedron to its integer points

Let

$$
P = \{x \in \mathbb{R}^n \mid Ax\le b\}
$$

be the feasible region of an LP relaxation. If every variable must be integral, the feasible set of the corresponding integer program is

$$
P\cap\mathbb{Z}^n.
$$

This is generally not convex: the line segment between two integer points contains fractional points. To recover a convex set without changing any linear optimization problem over the integer points, we take their convex hull.

The **integer hull** of $P$ is

$$
\boxed{P_I = \operatorname{conv}(P\cap\mathbb{Z}^n).}
$$

Because $P$ is convex and contains every point in $P\cap\mathbb{Z}^n$, we always have

$$
P_I \subseteq P.
$$

Moreover, no integer feasible points are lost:

$$
P_I\cap\mathbb{Z}^n = P\cap\mathbb{Z}^n.
$$

## Mixed-integer hulls

Sometimes only the first $p$ variables must be integral. The relevant feasible set is then

$$
P\cap\left(\mathbb{Z}^p\times\mathbb{R}^{n-p}\right),
$$

and its **mixed-integer hull** is

$$
P_{I,p}
= \operatorname{conv}\!\left(
P\cap\left(\mathbb{Z}^p\times\mathbb{R}^{n-p}\right)
\right).
$$

Two special cases help orient the notation:

- $p=0$: no integrality restrictions, so $P_{I,0}=P$;
- $p=n$: all variables are integral, so $P_{I,n}=P_I$.

We focus on the pure-integer case in the geometric example below.

## A two-dimensional example

Consider the rational polytope

$$
P = \left\{x\in\mathbb{R}^2\ \middle|\
\begin{aligned}
x_1 &\ge 0, \\
x_2 &\ge 0, \\
2x_1+x_2 &\le \frac{17}{2}, \\
x_1+2x_2 &\le \frac{17}{2}
\end{aligned}
\right\}.
$$

Its vertices are

$$
(0,0),\quad
\left(\frac{17}{4},0\right),\quad
\left(\frac{17}{6},\frac{17}{6}\right),\quad
\left(0,\frac{17}{4}\right).
$$

Three of the four vertices are fractional. The integer program does not permit those points, but it does permit every lattice point inside $P$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src import (
    hrep_to_vertices_2d,
    integer_hull_vertices_2d,
    lattice_points_in_polytope_2d,
)

A = np.array(
    [
        [-1.0, 0.0],       # x1 >= 0
        [0.0, -1.0],       # x2 >= 0
        [2.0, 1.0],        # 2*x1 + x2 <= 17/2
        [1.0, 2.0],        # x1 + 2*x2 <= 17/2
    ]
)
b = np.array([0.0, 0.0, 8.5, 8.5]).reshape(-1, 1)

polytope_vertices = hrep_to_vertices_2d(A, b)
lattice_points = lattice_points_in_polytope_2d(A, b)
integer_hull_vertices = integer_hull_vertices_2d(A, b)

print(f"Number of integer feasible points: {len(lattice_points)}")
print("Integer-hull vertices:")
print(integer_hull_vertices)

In [ ]:
def close_polygon(vertices):
    return np.vstack([vertices, vertices[0]])

P_closed = close_polygon(polytope_vertices)
PI_closed = close_polygon(integer_hull_vertices)

fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(P_closed[:, 0], P_closed[:, 1], color="tab:blue", alpha=0.15, label="$P$ (LP relaxation)")
ax.plot(P_closed[:, 0], P_closed[:, 1], color="tab:blue")
ax.fill(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", alpha=0.25, label="$P_I$ (integer hull)")
ax.plot(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", linewidth=2)
ax.scatter(lattice_points[:, 0], lattice_points[:, 1], color="black", s=25, zorder=3, label=r"$P\cap\mathbb{Z}^2$")
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("LP relaxation, integer points, and integer hull")
ax.grid(True)
ax.legend()
plt.show()

The picture contains three different objects:

- the blue polygon $P$, which is the continuous LP relaxation;
- the black lattice points $P\cap\mathbb{Z}^2$, which form the discrete feasible set;
- the green polygon $P_I$, which is their convex hull.

The integer hull is convex like the LP relaxation, but it has exactly the same integer points as the original discrete set. In this example,

$$
P_I = \operatorname{conv}\{(0,0),(4,0),(3,2),(2,3),(0,4)\}.
$$

## Computing the example by enumeration

For a bounded 2D polytope, a direct educational algorithm is enough:

1. Compute the vertices of $P$ and use them to obtain a finite integer bounding box.
2. Enumerate every lattice point in that box.
3. Keep the points satisfying $Ax\le b$.
4. Compute the convex hull of those feasible lattice points.

The first helper performs steps 1–3.

### Enumerating feasible lattice points

```{literalinclude} ../src/polyhedra/integer_hull.py
:language: python
:linenos:
:start-after: "# --- INTEGER_HULL:lattice_points:start"
:end-before: "# --- INTEGER_HULL:lattice_points:end"
```

The second helper performs step 4. It also handles the lower-dimensional cases where the integer hull is only one point or a line segment.

### Extracting the integer-hull vertices

```{literalinclude} ../src/polyhedra/integer_hull.py
:language: python
:linenos:
:start-after: "# --- INTEGER_HULL:vertices:start"
:end-before: "# --- INTEGER_HULL:vertices:end"
```

### Why this does not scale

If the bounding box contains $N_1$ possible values for $x_1$ and $N_2$ for $x_2$, enumeration checks $N_1N_2$ points. In $n$ dimensions this becomes roughly

$$
N_1N_2\cdots N_n,
$$

which grows exponentially with dimension. Large coordinate ranges create the same problem even in low dimensions.

The helper is therefore useful for visualization, examples, and verification—not as a general integer-hull algorithm.

## Why optimizing over the integer hull is equivalent

Let $c\in\mathbb{R}^n$ and assume the maximum is finite. A linear objective cannot improve when we replace a set by its convex hull:

$$
\max\{c^\top x\mid x\in P\cap\mathbb{Z}^n\}
=
\max\{c^\top x\mid x\in P_I\}.
$$

This matters because the left-hand side is optimization over a discrete set, while the right-hand side is a linear program over a convex polyhedron.

If we had a compact H-representation of $P_I$, we could solve the integer program by solving one LP. The difficulty is that finding this representation can be as hard as solving the integer program, and the integer hull may have very many facets.

## The relaxation gap and a valid inequality

For the example, maximize

$$
c^\top x=x_1+x_2.
$$

The LP relaxation chooses the intersection of the two slanted constraints:

$$
x^{LP}=\left(\frac{17}{6},\frac{17}{6}\right),
\qquad
c^\top x^{LP}=\frac{17}{3}\approx5.67.
$$

That solution is fractional. The best integer solutions are $(3,2)$ and $(2,3)$, both with objective value $5$.

In [ ]:
from scipy.optimize import linprog

c = np.array([1.0, 1.0])
lp_result = linprog(-c, A_ub=A, b_ub=b.ravel(), bounds=[(None, None)] * 2, method="highs")
integer_values = lattice_points @ c
best_integer_value = integer_values.max()
best_integer_points = lattice_points[np.isclose(integer_values, best_integer_value)]

print("LP optimum:", lp_result.x, "with value", -lp_result.fun)
print("Best integer points:", best_integer_points.tolist())
print("Integer optimum value:", best_integer_value)
print("Additive relaxation gap:", -lp_result.fun - best_integer_value)

Every integer feasible point satisfies the stronger inequality

$$
x_1+x_2\le5.
$$

The fractional LP optimum violates it because $17/3>5$. Adding this inequality removes the current LP optimum without removing any integer feasible point. Such an inequality is called a **valid inequality**; when it removes a fractional relaxation solution, it acts as a **cut**.

In this example, the inequality also supports an edge of $P_I$, so it defines a facet of the two-dimensional integer hull.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(P_closed[:, 0], P_closed[:, 1], color="tab:blue", alpha=0.15, label="$P$")
ax.plot(P_closed[:, 0], P_closed[:, 1], color="tab:blue")
ax.fill(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", alpha=0.25, label="$P_I$")
ax.plot(PI_closed[:, 0], PI_closed[:, 1], color="tab:green", linewidth=2)
ax.scatter(*lp_result.x, marker="x", s=100, linewidths=2, color="tab:red", label="LP optimum")
ax.scatter(best_integer_points[:, 0], best_integer_points[:, 1], marker="*", s=130, color="black", label="integer optima")
cut_x = np.linspace(0, 5, 100)
ax.plot(cut_x, 5 - cut_x, linestyle="--", color="tab:orange", label="$x_1+x_2=5$")
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("A valid inequality cuts off the fractional LP optimum")
ax.grid(True)
ax.legend()
plt.show()

For a maximization problem, the objective values satisfy

$$
z_{IP}=z_{P_I}\le z_{LP}.
$$

The difference $z_{LP}-z_{IP}$ is an **additive integrality gap** for this instance and objective. It measures how optimistic the LP relaxation is. Stronger relaxations move the LP bound closer to the integer optimum.

## Rational and integral polyhedra

A polyhedron is **rational** if it has an H-representation

$$
P=\{x\mid Ax\le b\}
$$

with rational $A$ and $b$. Rationality is fundamental because integer programs are normally encoded using finite rational data.

A polyhedron is **integral** if

$$
P=P_I.
$$

For a rational polytope, this is equivalent to saying that every vertex is integral. The polytope in our example is not integral because it has fractional vertices.

Two useful facts are:

1. If $P$ is a rational polyhedron, then $P_I$ is also a rational polyhedron.
2. If a rational polytope has only integral vertices, optimizing an LP over it automatically produces an integral vertex optimum whenever an optimum exists at a vertex.

The second fact is why integral formulations are so valuable. Later, total unimodularity will give us a structural way to recognize important families of integral polyhedra.

## How algorithms use this idea

Explicitly constructing $P_I$ is usually impractical. Modern integer-optimization algorithms instead reveal enough of its structure while solving:

- **Cutting-plane methods** add valid inequalities that approximate facets of $P_I$ and remove fractional LP solutions.
- **Branch-and-bound** divides the relaxation into smaller regions until fractional solutions are excluded or bounded.
- **Branch-and-cut** combines both approaches.

From this viewpoint, integer optimization repeatedly improves a tractable outer approximation

$$
P_I \subseteq Q \subseteq P
$$

until the relaxation is strong enough to certify an optimal integer solution.

## Takeaways

- The integer feasible set $P\cap\mathbb{Z}^n$ is discrete and generally non-convex.
- Its convex hull $P_I=\operatorname{conv}(P\cap\mathbb{Z}^n)$ is the **integer hull**.
- Linear optimization over $P_I$ is equivalent to linear optimization over the integer feasible points.
- The LP relaxation $P$ can have a better fractional optimum, creating an integrality gap.
- Valid inequalities preserve all integer feasible points while tightening the relaxation; facets of $P_I$ are the strongest geometric target.
- Rational polyhedra have rational polyhedral integer hulls, but those hulls may be difficult or exponentially large to describe.
- Small 2D integer hulls can be computed by lattice-point enumeration, while practical solvers use branching and cutting instead.

## Next

The next notebook develops the first major strategy for avoiding an explicit integer-hull description: recursively split the LP relaxation and use bounds to discard regions that cannot contain a better integer solution.

➡️ [06 - Branch and Bound](06-Branch-And-Bound.ipynb)